Import Libraries and Configure Logging

In [77]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from gensim.corpora import Dictionary
from gensim.models import LdaModel, CoherenceModel
from gensim.models.phrases import Phraser
from tqdm import tqdm
import logging
import warnings

# Setup logging
logging.basicConfig(
    filename='lda_inference.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logging.getLogger('gensim').setLevel(logging.WARNING)

# Download NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

True

Initialize NLP Tools and Stopwords

In [78]:
# Initialize lemmatizer and stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Add custom stopwords
custom_stopwords = {
    'allow', 'class', 'available', 'part', 'case', 'lead', 'shall', 'product', 'operate',
    'operational', 'result', 'input', 'dependent', 'preference', 'item', 'without', 'let',
    'returned', 'message', 'every', 'system', 'run', 'fully', 'major', 'reasonable',
    'software', 'user', 'able', 'ability', 'support', 'year', 'expected', 'must',
    'information', 'data', 'use', 'using', 'provide', 'successfully', 'one', 'waiter',
    'include', 'accommodate', 'event', 'technique', 'recent', 'administrator', 'search',
    'add', 'allows', 'achieve', 'way', 'outside', 'release', 'launch', 'allowed', 'entered',
    'within', 'first', 'new', 'izogn', 'wcs', 'course', 'time', 'help', 'learn', 'ccr', 'cma',
    'review', 'star', 'rating', 'good', 'great', 'course', 'learn', 'learning',
    'would', 'like', 'could', 'one', 'bit', 'week', 'think', 'much', 'really',
    'lot', 'new', 'thank', 'thanks', 'many', 'well', 'also', 'get', 'time',
    'truly', 'even', 'make', 'see', 'content', 'material', 'class', 'work',
    'way', 'understand', 'information', 'helpful', 'useful', 'knowledge',
    'day', 'help', 'easy'
}
stop_words.update(custom_stopwords)

Load Seed Words

In [79]:
def load_seed_words_from_csv(file_path):
    """Load seed words from CSV file and return as dictionary"""
    try:
        df = pd.read_csv(file_path)
        seed_words = df.groupby('Category')['SeedWord'].apply(list).to_dict()
        logging.info(f"Successfully loaded seed words from {file_path}")
        return seed_words
    except FileNotFoundError:
        logging.error(f"Seed words file not found at {file_path}")
        raise
    except Exception as e:
        logging.error(f"Error loading seed words: {e}")
        raise

# Load seed words and update stopwords
seed_words_path = '../../datasets/seed_words.csv'
seed_words = load_seed_words_from_csv(seed_words_path)
seed_word_set = set(word for words in seed_words.values() for word in words)
stop_words = stop_words - seed_word_set

print("Seed words loaded successfully!")
print(f"Number of seed word categories: {len(seed_words)}")

Seed words loaded successfully!
Number of seed word categories: 7


Define Preprocessing Function

In [80]:
def preprocess(text, stop_words, lemmatizer, bigram_phraser):
    """Preprocess text for LDA model"""
    if not isinstance(text, str) or not text.strip():
        return []
    
    # Convert to lowercase and remove non-alphabetic characters
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Tokenize and lemmatize
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(word) for word in tokens 
              if word not in stop_words and len(word) > 2]
    
    # Apply bigram phraser
    tokens = bigram_phraser[tokens]
    return tokens

Define Coherence Evaluation Function

In [81]:
def compute_coherence_range_robust(lda_model, tokenized_reviews, dictionary, id_to_topic, window_size=50, topn_range=(10, 50, 10)):
    """
    Compute coherence scores for multiple topn values with robust error handling
    """
    logging.info("Started computing C_v coherence scores for multiple topn values")
    print("\n--- Computing C_v Coherence for Multiple topn Values (Robust) ---")
    
    start, end, step = topn_range
    coherence_results = []
    
    # Pre-filter topics to avoid problematic ones
    valid_topics = []
    for topic_id in range(lda_model.num_topics):
        topic_words = lda_model.show_topic(topic_id, topn=20)
        # Check if topic has reasonable word probabilities
        if len(topic_words) > 0 and topic_words[0][1] > 0.01:  # At least 1% probability for top word
            valid_topics.append(topic_id)
    
    print(f"Valid topics for coherence evaluation: {valid_topics}")
    
    for topn in range(start, end + 1, step):
        logging.info(f"Computing C_v coherence for topn={topn}")
        print(f"\nComputing C_v Coherence for topn={topn}")
        
        try:
            # Suppress warnings for cleaner output
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                
                coherence_model = CoherenceModel(
                    model=lda_model,
                    texts=tokenized_reviews,
                    dictionary=dictionary,
                    coherence='c_v',
                    topn=min(topn, len(dictionary) // 2),  # Limit topn to reasonable size
                    window_size=window_size
                )
                
                per_topic_coherence = coherence_model.get_coherence_per_topic()
                
                # Handle NaN values
                valid_coherences = []
                print(f"\nPer-Topic C_v Scores (topn={topn}):")
                for i, score in enumerate(per_topic_coherence):
                    topic_name = id_to_topic[i]
                    if np.isnan(score):
                        logging.warning(f"Topic #{i} ({topic_name}): C_v Score = NaN (skipped)")
                        print(f"Topic #{i}: {topic_name} - C_v Score = NaN (skipped)")
                    else:
                        valid_coherences.append(score)
                        logging.info(f"Topic #{i} ({topic_name}): C_v Score = {score:.4f} (topn={topn})")
                        print(f"Topic #{i}: {topic_name} - C_v Score = {score:.4f}")
                
                # Compute overall coherence from valid scores only
                if valid_coherences:
                    overall_coherence = np.mean(valid_coherences)
                    logging.info(f"Overall C_v Coherence Score (topn={topn}): {overall_coherence:.4f}")
                    print(f"\nOverall C_v Coherence Score (topn={topn}): {overall_coherence:.4f}")
                    print(f"Valid topics: {len(valid_coherences)}/{len(per_topic_coherence)}")
                else:
                    overall_coherence = np.nan
                    logging.warning(f"No valid coherence scores for topn={topn}")
                    print(f"\nNo valid coherence scores for topn={topn}")
                
                print("-" * 50)
                
                coherence_results.append({
                    'topn': topn,
                    'per_topic_coherence': per_topic_coherence,
                    'overall_coherence': overall_coherence,
                    'valid_topics': len(valid_coherences),
                    'total_topics': len(per_topic_coherence)
                })
                
        except Exception as e:
            logging.error(f"Error computing coherence for topn={topn}: {e}")
            print(f"Error computing coherence for topn={topn}: {e}")
            continue
    
    logging.info("Completed computing C_v coherence scores for multiple topn values")
    print("\n--- Completed Computing C_v Coherence for Multiple topn Values ---")
    return coherence_results

def analyze_coherence_results(coherence_results):
    """
    Analyze and visualize coherence results
    """
    print("\n=== COHERENCE ANALYSIS SUMMARY ===")
    
    # Create summary DataFrame
    summary_data = []
    for result in coherence_results:
        summary_data.append({
            'topn': result['topn'],
            'overall_coherence': result['overall_coherence'],
            'valid_topics': result['valid_topics'],
            'total_topics': result['total_topics'],
            'success_rate': result['valid_topics'] / result['total_topics']
        })
    
    summary_df = pd.DataFrame(summary_data)
    
    print("\nCoherence Summary:")
    print(summary_df.to_string(index=False, float_format='%.4f'))
    
    # Find optimal topn
    valid_results = summary_df[~summary_df['overall_coherence'].isna()]
    if not valid_results.empty:
        best_topn = valid_results.loc[valid_results['overall_coherence'].idxmax(), 'topn']
        best_coherence = valid_results['overall_coherence'].max()
        print(f"\nBest coherence: {best_coherence:.4f} at topn={best_topn}")
    
    return summary_df

Set Input Parameters

In [82]:
# Define input parameters
class Args:
    input = '../../datasets/review_test.csv'

args = Args()
print(f"Input file: {args.input}")

Input file: ../../datasets/review_test.csv


Load Pre-trained Models

In [83]:
# Load saved models
logging.info("Loading saved models")
try:
    lda_model = LdaModel.load('../../train/Model_LDA/models/lda_model')
    dictionary = Dictionary.load('../../train/Model_LDA/models/dictionary')
    bigram_phraser = Phraser.load('../../train/Model_LDA/models/bigram_phraser')
    logging.info("Loaded LDA model, dictionary, and bigram model")
    print("✓ Successfully loaded all models")
    print(f"  - Number of topics: {lda_model.num_topics}")
    print(f"  - Dictionary size: {len(dictionary)}")
except FileNotFoundError as e:
    logging.error(f"Failed to load model files: {e}")
    print(f"\nError: Failed to load model files: {e}")
    raise

✓ Successfully loaded all models
  - Number of topics: 7
  - Dictionary size: 15789


Load and Prepare Dataset

In [84]:
# Load new dataset
logging.info("Loading new dataset")
try:
    df = pd.read_csv(args.input)
    df = df[['processed_reviews']].copy()
    df['processed_reviews'] = df['processed_reviews'].fillna("")
    logging.info("Loaded new dataset")
    print(f"✓ Successfully loaded dataset with {len(df)} reviews")
except FileNotFoundError as e:
    logging.error(f"Failed to load dataset {args.input}: {e}")
    print(f"\nError: Failed to load dataset {args.input}: {e}")
    raise

✓ Successfully loaded dataset with 71630 reviews


Preprocess Reviews

In [85]:
# Preprocess dataset
logging.info("Preprocessing new dataset")
tqdm.pandas()
tokenized_reviews = df['processed_reviews'].progress_apply(
    lambda x: preprocess(x, stop_words, lemmatizer, bigram_phraser)
)

# Filter out empty reviews
valid_indices = [i for i, tokens in enumerate(tokenized_reviews) if tokens]
tokenized_reviews = [tokens for tokens in tokenized_reviews if tokens]

if not tokenized_reviews:
    raise ValueError("No valid reviews after preprocessing.")

filtered_df = df.iloc[valid_indices].copy()
logging.info(f"Filtered to {len(filtered_df)} valid reviews")
print(f"✓ Preprocessed {len(tokenized_reviews)} valid reviews")

100%|██████████| 71630/71630 [00:02<00:00, 24143.86it/s]

✓ Preprocessed 70412 valid reviews


Create Bag-of-Words Corpus

In [86]:
# Create BoW corpus
logging.info("Creating BoW corpus")
corpus = [dictionary.doc2bow(text) for text in tokenized_reviews]
logging.info("Created BoW corpus")
print(f"✓ Created corpus with {len(corpus)} documents")

✓ Created corpus with 70412 documents


Infer Topics

In [87]:
# Infer topics
logging.info("Inferring topics")
topic_matrix = np.zeros((len(corpus), lda_model.num_topics))

for i, doc in enumerate(corpus):
    topics = lda_model.get_document_topics(doc, minimum_probability=0.0)
    for topic_id, prob in topics:
        topic_matrix[i, topic_id] = prob

topic_assignments = np.argmax(topic_matrix, axis=1)
logging.info("Completed topic inference")
print(f"✓ Completed topic inference for {len(corpus)} documents")

✓ Completed topic inference for 70412 documents


Define Topic Mappings

In [88]:
# Define topic mappings
topic_name_to_id = {name: idx for idx, name in enumerate(seed_words.keys())}
id_to_topic = {v: k for k, v in topic_name_to_id.items()}

print("\nTopic Name to Integer ID Mapping:")
for name, id in topic_name_to_id.items():
    print(f"  {name}: {id}")


Topic Name to Integer ID Mapping:
  F: 0
  FT: 1
  PE: 2
  PO: 3
  SC: 4
  SE: 5
  US: 6


Diagnose Topic Issues

Compute Coherence Scores

In [89]:
# Compute coherence scores with robust method
# Using reduced topn range to avoid numerical issues
coherence_results = compute_coherence_range_robust(
    lda_model=lda_model,
    tokenized_reviews=tokenized_reviews,
    dictionary=dictionary,
    id_to_topic=id_to_topic,
    window_size=50,
    topn_range=(10, 60, 10)  # Reduced upper limit to avoid numerical issues
)

# Analyze results
summary_df = analyze_coherence_results(coherence_results)


--- Computing C_v Coherence for Multiple topn Values (Robust) ---
Valid topics for coherence evaluation: [0, 1, 2, 3, 5, 6]

Computing C_v Coherence for topn=10

Per-Topic C_v Scores (topn=10):
Topic #0: F - C_v Score = 0.4239
Topic #1: FT - C_v Score = 0.4422
Topic #2: PE - C_v Score = 0.5267
Topic #3: PO - C_v Score = 0.3407
Topic #4: SC - C_v Score = 0.6569
Topic #5: SE - C_v Score = 0.5762
Topic #6: US - C_v Score = 0.4369

Overall C_v Coherence Score (topn=10): 0.4862
Valid topics: 7/7
--------------------------------------------------

Computing C_v Coherence for topn=20

Per-Topic C_v Scores (topn=20):
Topic #0: F - C_v Score = 0.2716
Topic #1: FT - C_v Score = 0.3681
Topic #2: PE - C_v Score = 0.4434
Topic #3: PO - C_v Score = 0.3147
Topic #4: SC - C_v Score = 0.6608
Topic #5: SE - C_v Score = 0.4976
Topic #6: US - C_v Score = 0.3356

Overall C_v Coherence Score (topn=20): 0.4131
Valid topics: 7/7
--------------------------------------------------

Computing C_v Coherence for 

Verify Seed Word Probabilities

In [90]:
# Verify seed word probabilities
logging.info("Verifying Seed Word Probabilities and Ranks in Final Model")
print("\n--- Verifying Seed Word Probabilities and Ranks in Final Model ---")

seed_word_ids = {
    topic: [dictionary.token2id[word] for word in words if word in dictionary.token2id]
    for topic, words in seed_words.items()
}

for topic_name, words in seed_words.items():
    topic_id = topic_name_to_id[topic_name]
    print(f"\nAssigned Topic: {topic_name} (ID: {topic_id})")
    logging.info(f"Assigned Topic: {topic_name} (ID: {topic_id})")
    
    for word in words:
        if word in dictionary.token2id:
            word_id = dictionary.token2id[word]
            term_topics = lda_model.get_term_topics(word_id, minimum_probability=0.0)
            print(f"  - Seed Word '{word}'")
            logging.info(f"  - Seed Word '{word}'")
            
            for t_id, prob in term_topics:
                t_name = id_to_topic[t_id]
                topic_words_probs = lda_model.show_topic(t_id, topn=len(dictionary))
                word_to_rank = {w: idx + 1 for idx, (w, _) in enumerate(topic_words_probs)}
                rank = word_to_rank.get(word, "N/A")
                log_message = f"    * Topic {t_name} (ID: {t_id}): Probability = {prob:.4f}, Rank = {rank}"
                logging.info(log_message)
                print(log_message)
        else:
            logging.warning(f"Seed Word '{word}' for topic '{topic_name}' was not in the final dictionary.")
            print(f"  - '{word}' (Not in dictionary)")

print("----------------------------------------------------------\n")


--- Verifying Seed Word Probabilities and Ranks in Final Model ---

Assigned Topic: F (ID: 0)
  - Seed Word 'player'
    * Topic F (ID: 0): Probability = 0.0046, Rank = 61
  - Seed Word 'display'
    * Topic F (ID: 0): Probability = 0.0048, Rank = 57
  - Seed Word 'meeting'
    * Topic F (ID: 0): Probability = 0.0051, Rank = 53
  - Seed Word 'dispute'
    * Topic F (ID: 0): Probability = 0.0046, Rank = 60
  - Seed Word 'program'
    * Topic F (ID: 0): Probability = 0.0180, Rank = 9
    * Topic SC (ID: 4): Probability = 0.0012, Rank = 210
  - Seed Word 'clinical'
    * Topic F (ID: 0): Probability = 0.0099, Rank = 18
  - Seed Word 'member'
    * Topic F (ID: 0): Probability = 0.0052, Rank = 50
  - Seed Word 'staff'
    * Topic F (ID: 0): Probability = 0.0120, Rank = 12
  - Seed Word 'student'
    * Topic F (ID: 0): Probability = 0.0193, Rank = 8
    * Topic SC (ID: 4): Probability = 0.0023, Rank = 80
  - Seed Word 'lab'
    * Topic F (ID: 0): Probability = 0.0207, Rank = 6
  - Seed Wor

Summary Results and Recommendations

In [91]:
# Display comprehensive summary results
print("\n=== COMPREHENSIVE SUMMARY RESULTS ===")
print(f"Total documents processed: {len(corpus)}")
print(f"Number of topics: {lda_model.num_topics}")
print(f"Dictionary size: {len(dictionary)}")

# Display coherence summary again
if 'summary_df' in locals():
    print("\nFinal Coherence Summary:")
    print(summary_df.to_string(index=False, float_format='%.4f'))
    
    # Best performing topn
    valid_results = summary_df[~summary_df['overall_coherence'].isna()]
    if not valid_results.empty:
        best_idx = valid_results['overall_coherence'].idxmax()
        best_row = valid_results.iloc[best_idx]
        print(f"\n📊 BEST PERFORMANCE:")
        print(f"   topn: {best_row['topn']}")
        print(f"   Coherence: {best_row['overall_coherence']:.4f}")
        print(f"   Success rate: {best_row['success_rate']:.2%}")

print("\n✓ Analysis completed successfully!")


=== COMPREHENSIVE SUMMARY RESULTS ===
Total documents processed: 70412
Number of topics: 7
Dictionary size: 15789

Final Coherence Summary:
 topn  overall_coherence  valid_topics  total_topics  success_rate
   10             0.4862             7             7        1.0000
   20             0.4131             7             7        1.0000
   30             0.3998             7             7        1.0000
   40             0.3905             7             7        1.0000
   50             0.3996             7             7        1.0000
   60             0.4296             7             7        1.0000

📊 BEST PERFORMANCE:
   topn: 10.0
   Coherence: 0.4862
   Success rate: 100.00%

✓ Analysis completed successfully!
